# File Transfer Protocol

In [0]:
from ftplib import FTP

ftp_host = "test.rebex.net"
ftp_user = "demo"
ftp_password = "password"

with FTP(ftp_host) as ftp:
    ftp.login(ftp_user, ftp_password)
    
    # Navegamos hacia la carpeta interna
    print("Navegando a /pub/example...")
    ftp.cwd("pub/example") 
    
    print(f"Directorio actual: {ftp.pwd()}")
    print("Contenido:")
    # pero retrlines('LIST') nos da más info (tamaño, fecha).
    ftp.retrlines("LIST")

In [0]:
from ftplib import FTP
import io

ftp_host = "test.rebex.net"
ftp_user = "demo"
ftp_password = "password"

# Creamos un contenedor en la memoria de Python para guardar el archivo
archivo_en_memoria = io.BytesIO()

with FTP(ftp_host) as ftp:
    ftp.login(ftp_user, ftp_password)
    
    # 1. Navegamos a la carpeta donde viste los archivos
    ftp.cwd("pub/example")
    
    # 2. Elegimos el archivo ESPECÍFICO que queremos leer (en este caso, readme.txt)
    nombre_archivo = "readme.txt"
    print(f"Descargando el archivo: {nombre_archivo}...")
    
    # 3. Ejecutamos el comando RETR (Retrieve/Recuperar) para ese archivo
    ftp.retrbinary(f"RETR {nombre_archivo}", archivo_en_memoria.write)

# 4. Una vez descargado, volvemos al inicio del contenedor virtual y lo leemos
archivo_en_memoria.seek(0)
contenido_texto = archivo_en_memoria.read().decode('utf-8')

print("\n--- CONTENIDO DEL ARCHIVO ---")
print(contenido_texto)
print("-----------------------------")

In [0]:
from ftplib import FTP
import io
# Importamos las herramientas del Notebook para mostrar imágenes
from IPython.display import Image, display

ftp_host = "test.rebex.net"
ftp_user = "demo"
ftp_password = "password"

# 1. Creamos nuestro contenedor en memoria
imagen_en_memoria = io.BytesIO()

# Elegimos una de las imágenes de tu lista
nombre_imagen = "imap-console-client.png"

with FTP(ftp_host) as ftp:
    ftp.login(ftp_user, ftp_password)
    ftp.cwd("pub/example")
    
    print(f"📥 Descargando los bytes de: {nombre_imagen}...")
    
    # 2. Ejecutamos el RETR para llenar nuestro contenedor con los bytes de la imagen
    ftp.retrbinary(f"RETR {nombre_imagen}", imagen_en_memoria.write)

print("¡Descarga completa!")

# 3. Extraemos los bytes puros del contenedor
datos_imagen = imagen_en_memoria.getvalue()

# 4. Le decimos al Notebook que interprete esos bytes y los muestre
print("\n--- MOSTRANDO IMAGEN ---")
display(Image(data=datos_imagen))

# SSH File Transfer Protocol

In [0]:
%pip install paramiko

In [0]:
import paramiko

host = "test.rebex.net"
user = "demo"
password = "password"

# Creamos el cliente de transporte SSH
transport = paramiko.Transport((host, 22))
transport.connect(username=user, password=password)

# Creamos la sesión SFTP sobre ese transporte
sftp = paramiko.SFTPClient.from_transport(transport)

print("Conectado a SFTP!")
print("Archivos:", sftp.listdir('.'))

sftp.close()
transport.close()

In [0]:
import paramiko

# Credenciales del servidor SFTP de pruebas
host = "test.rebex.net"
user = "demo"
password = "password"

# 1. Creamos el "túnel" de transporte SSH (Puerto 22)
transport = paramiko.Transport((host, 22))
transport.connect(username=user, password=password)

# 2. Iniciamos la sesión SFTP a través de ese túnel
sftp = paramiko.SFTPClient.from_transport(transport)
print("Conectado a SFTP de Rebex de forma segura\n")

# 3. Leer el archivo de texto
ruta_texto = "pub/example/readme.txt"
print(f"📄 Leyendo: {ruta_texto}...\n")

# Usamos sftp.open en modo lectura ('r')
with sftp.open(ruta_texto, 'r') as archivo_remoto:
    # Leemos el contenido y lo decodificamos a texto
    contenido = archivo_remoto.read().decode('utf-8')
    print("--- CONTENIDO ---")
    print(contenido)
    print("-----------------")

In [0]:
from IPython.display import Image, display

ruta_imagen = "pub/example/imap-console-client.png"
print(f" Descargando bytes de la imagen: {ruta_imagen}...\n")

# Usamos sftp.open en modo lectura binaria ('rb')
with sftp.open(ruta_imagen, 'rb') as archivo_remoto:
    # A diferencia del texto, aquí NO decodificamos, solo extraemos los bytes puros
    datos_imagen = archivo_remoto.read()

print("Imagen procesada en memoria. Mostrando:")

# Le pedimos al Notebook que dibuje esos bytes en la pantalla
display(Image(data=datos_imagen))

In [0]:
# Cerramos primero la sesión de archivos (SFTP)
sftp.close()

# Cerramos finalmente el túnel de comunicación (SSH)
transport.close()

print("🔒 Conexión SFTP cerrada exitosamente.")

In [0]:
import paramiko

# Credenciales (SFTP usa el puerto 22 por defecto)
sftp_host = "test.rebex.net"
sftp_user = "demo"
sftp_password = "password"

# 1. ESTABLECER LA CONEXIÓN SSH/SFTP
# A diferencia de ftplib, aquí creamos un "Transporte" y luego abrimos la sesión
transport = paramiko.Transport((sftp_host, 22))
transport.connect(username=sftp_user, password=sftp_password)

sftp = paramiko.SFTPClient.from_transport(transport)

try:
    # 2. NAVEGAR A LA CARPETA
    # En SFTP usamos 'chdir' (Change Directory) en lugar de 'cwd'
    print("Navegando a /pub/example...")
    sftp.chdir("pub/example")
    
    # 3. DIRECTORIO ACTUAL
    # Usamos 'getcwd' (Get Current Working Directory) en lugar de 'pwd'
    print(f"Directorio actual: {sftp.getcwd()}")
    
    # 4. LISTAR CONTENIDO CON DETALLES
    print("Contenido:")
    # 'listdir_attr()' es el equivalente a 'LIST'. 
    # Nos devuelve objetos con todos los metadatos (tamaño, fecha, permisos).
    # Al imprimir el atributo '.longname', obtenemos esa vista detallada que querías.
    for archivo in sftp.listdir_attr():
        print(archivo.longname)

finally:
    # 5. CERRAR CONEXIONES
    # Es vital usar un bloque try/finally (o un 'with' si usáramos otra librería) 
    # para asegurar que el túnel se cierre, incluso si hay un error.
    sftp.close()
    transport.close()
    print("\nConexión cerrada.")

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.raw.rebex
""")

In [0]:
# 1. Eliminamos la carpeta y TODO su contenido (el 'True' significa eliminación recursiva)
dbutils.fs.rm("/Volumes/workspace/raw/rebex", True)

In [0]:
import paramiko
import os

# 1. CONFIGURACIÓN
sftp_host = "test.rebex.net"
sftp_user = "demo"
sftp_password = "password"
carpeta_remota = "pub/example"
ruta_volumen_databricks = "/Volumes/workspace/raw/rebex"

os.makedirs(ruta_volumen_databricks, exist_ok=True)

In [0]:
print(f"Conectando a {sftp_host}...")
transport = paramiko.Transport((sftp_host, 22))
transport.connect(username=sftp_user, password=sftp_password)
sftp = paramiko.SFTPClient.from_transport(transport)

try:
    sftp.chdir(carpeta_remota)
    lista_archivos = sftp.listdir()
    
    print(f"Se encontraron {len(lista_archivos)} archivos en el SFTP. Evaluando...\n")
    
    # Contadores para nuestro log final
    descargados = 0
    omitidos = 0

    # 4. BUCLE DE DESCARGA INCREMENTAL
    for nombre_archivo in lista_archivos:
        ruta_destino = os.path.join(ruta_volumen_databricks, nombre_archivo)
        
        # EL CAMBIO CLAVE: Verificamos si el archivo ya vive en nuestro Volumen
        if os.path.exists(ruta_destino):
            print(f" Omitiendo: {nombre_archivo} (Ya existe en destino)")
            omitidos += 1
        else:
            print(f"Descargando: {nombre_archivo}...")
            sftp.get(nombre_archivo, ruta_destino)
            descargados += 1

    print(f"\n Resumen de ingesta: {descargados} nuevos descargados, {omitidos} omitidos.")

finally:
    sftp.close()
    transport.close()
    print("Conexión SFTP cerrada.")